# DSA 2050
## Week 2 Practical Lab
### SQL, Data Acquisition and Join Validation

**Student:** Tamara Mwakisha  
**Student ID:** 660146

This lab examines customer and order records, checks their keys, and tests the effect of joining them. The source records are retained until the duplicate and unmatched keys have been investigated.

## Part A: Create the Source Data
Create the three source files exactly as specified in the practical. Keep the duplicate customer record and unmatched order in the files for investigation.

In [1]:
import pandas as pd
import sqlite3
import json

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

customers = pd.DataFrame({
    "CustomerID": ["C001", "C002", "C003", "C004", "C004", "C005", "C006", "C007", "C008", "C009", "C010"],
    "CustomerName": ["Amina", "Brian", "Carol", "David", "David", "Esther", "Faith", "George", "Halima", "Ian", "Jane"],
    "Segment": ["Retail", "Corporate", "Retail", "SME", "SME", "Corporate", "SME", "Retail", "Corporate", "Retail", "SME"],
    "RegionCode": ["NRB", "MSA", "NRB", "KSM", "KSM", "NKR", "NRB", "MSA", "KSM", "NKR", "NRB"]
})

orders = pd.DataFrame({
    "OrderID": [f"O{i:03d}" for i in range(1, 16)],
    "CustomerID": ["C001", "C002", "C001", "C003", "C004", "C005", "C006", "C002", "C007", "C008", "C009", "C010", "C006", "C003", "C999"],
    "OrderAmount": [4500, 12500, 3200, 6800, 9100, 15000, 7600, 8300, 2500, 11000, 5400, 6200, 4300, 7200, 9900],
    "ProductCategory": ["Electronics", "Office", "Home", "Electronics", "Office", "Electronics", "Home", "Office", "Home", "Electronics", "Office", "Home", "Electronics", "Office", "Electronics"]
})

regions = [
    {"RegionCode": "NRB", "RegionName": "Nairobi"},
    {"RegionCode": "MSA", "RegionName": "Mombasa"},
    {"RegionCode": "KSM", "RegionName": "Kisumu"},
    {"RegionCode": "NKR", "RegionName": "Nakuru"}
]

customers.to_csv("customers.csv", index=False)
orders.to_csv("orders.csv", index=False)
with open("regions.json", "w") as file:
    json.dump(regions, file, indent=4)

print("Source files created.")

Source files created.


## Part B: Inspect the Sources

**Q1.** One row in `customers.csv` represents a customer record. The intended grain is one row per customer ID. The file contains 11 rows but only 10 distinct IDs because `C004` appears twice.

**Q2.** One row in `orders.csv` represents one order, identified by `OrderID`. There are 15 order records.

**Q3.** The datasets are not at the same grain. The customer file is at customer grain, while the orders file is at transaction grain. A customer can have several orders, so each order should match no more than one customer record in a valid customer lookup.

In [2]:
customers = pd.read_csv("customers.csv")
orders = pd.read_csv("orders.csv")

print("Customer shape:", customers.shape)
print("Order shape:", orders.shape)
display(customers.head())
display(orders.head())

Customer shape: (11, 4)
Order shape: (15, 4)


,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O002,C002,12500,Office
2,O003,C001,3200,Home
3,O004,C003,6800,Electronics
4,O005,C004,9100,Office


**Q1.** One row in `customers.csv` represents one customer record. `C004` appears twice, so the raw source has a duplicate customer record.

**Q2.** One row in `orders.csv` represents one order, identified by `OrderID`.

**Q3.** The datasets are not at the same grain. The customer file has one record per customer, while the order file has one record per transaction. One customer can have several orders, so the files are at different grains.

## Part C: Detect Key Problems

**Q4.** A duplicated `CustomerID` on the customer side makes the join one to many where it should be many to one. Every order for that ID can match both customer records, repeating the order and inflating row counts and sales.

**Q5.** `C999` appears in the orders but not in the customer file.

**Q6.** The missing customer record could be a data entry error, a late customer record, or an issue with how the sources were extracted. It should be checked before any correction is made. Silently deleting the order would remove `O015` and its `Ksh. 9,900` from the sales analysis.

In [3]:
duplicate_count = int(customers["CustomerID"].duplicated().sum())
duplicate_customers = customers.loc[
    customers["CustomerID"].duplicated(keep=False)
].sort_values(["CustomerID", "CustomerName"])

customer_ids = set(customers["CustomerID"])
order_customer_ids = set(orders["CustomerID"])
unmatched_customer_ids = sorted(order_customer_ids - customer_ids)

print(f"Repeated CustomerID values after first occurrence: {duplicate_count}")
display(duplicate_customers)
print("CustomerIDs present in orders but absent from customers:", unmatched_customer_ids)

Repeated CustomerID values after first occurrence: 1


,CustomerID,CustomerName,Segment,RegionCode
3,C004,David,SME,KSM
4,C004,David,SME,KSM


CustomerIDs present in orders but absent from customers: ['C999']


**Q4.** A duplicate `CustomerID` lets one order match more than one customer row. The join can therefore repeat orders and overstate counts and sales.

**Q5.** `C999` is present in the orders but absent from the customer file.

**Q6.** The missing customer record may be a load problem, a recording error, or a valid order awaiting identification. Deleting it without checking would remove an order worth `Ksh. 9,900`.

## Part D: Create the SQLite Database
Load the original customer and order data into SQLite.

In [4]:
conn = sqlite3.connect("retail_lab.db")
customers.to_sql("customers", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)

test_query = "SELECT * FROM customers LIMIT 5;"
database_sample = pd.read_sql_query(test_query, conn)
assert len(database_sample) == 5
assert {"customers", "orders"}.issubset(
    set(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)["name"])
)
display(database_sample)
print("SQLite tables verified: customers, orders.")

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


SQLite tables verified: customers, orders.


## Part E: `SELECT` and `WHERE`
Use SQL to retrieve and filter the requested records.

In [5]:
# E1   Return only the specified order fields.
select_query = """
SELECT OrderID, CustomerID, OrderAmount
FROM orders
ORDER BY OrderID;
"""
selected_orders = pd.read_sql_query(select_query, conn)
display(selected_orders.assign(OrderAmount=selected_orders["OrderAmount"].map(lambda x: f"Ksh. {x:,.0f}")))

# E2   Apply each lab filter independently.
retail_query = "SELECT CustomerID, CustomerName, Segment, RegionCode FROM customers WHERE Segment = 'Retail' ORDER BY CustomerID;"
over_8000_query = "SELECT OrderID, CustomerID, OrderAmount, ProductCategory FROM orders WHERE OrderAmount > 8000 ORDER BY OrderID;"
electronics_query = "SELECT OrderID, CustomerID, OrderAmount FROM orders WHERE ProductCategory = 'Electronics' ORDER BY OrderID;"
electronics_over_8000_query = """
SELECT OrderID, CustomerID, OrderAmount
FROM orders
WHERE ProductCategory = 'Electronics' AND OrderAmount > 8000
ORDER BY OrderID;
"""

retail_result = pd.read_sql_query(retail_query, conn)
over_8000_result = pd.read_sql_query(over_8000_query, conn)
electronics_result = pd.read_sql_query(electronics_query, conn)
electronics_over_8000_result = pd.read_sql_query(electronics_over_8000_query, conn)

def display_money(df, columns):
    shown = df.copy()
    for column in columns:
        shown[column] = shown[column].map(lambda value: f"Ksh. {value:,.0f}")
    display(shown)

print("Retail-segment customers")
display(retail_result)
print("Orders above Ksh. 8,000")
display_money(over_8000_result, ["OrderAmount"])
print("Electronics orders")
display_money(electronics_result, ["OrderAmount"])
print("Electronics orders above Ksh. 8,000")
display_money(electronics_over_8000_result, ["OrderAmount"])

,OrderID,CustomerID,OrderAmount
0,O001,C001,"Ksh. 4,500"
1,O002,C002,"Ksh. 12,500"
2,O003,C001,"Ksh. 3,200"
3,O004,C003,"Ksh. 6,800"
4,O005,C004,"Ksh. 9,100"
5,O006,C005,"Ksh. 15,000"
6,O007,C006,"Ksh. 7,600"
7,O008,C002,"Ksh. 8,300"
8,O009,C007,"Ksh. 2,500"
9,O010,C008,"Ksh. 11,000"


Retail-segment customers


,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C003,Carol,Retail,NRB
2,C007,George,Retail,MSA
3,C009,Ian,Retail,NKR


Orders above Ksh. 8,000


,OrderID,CustomerID,OrderAmount,ProductCategory
0,O002,C002,"Ksh. 12,500",Office
1,O005,C004,"Ksh. 9,100",Office
2,O006,C005,"Ksh. 15,000",Electronics
3,O008,C002,"Ksh. 8,300",Office
4,O010,C008,"Ksh. 11,000",Electronics
5,O015,C999,"Ksh. 9,900",Electronics


Electronics orders


,OrderID,CustomerID,OrderAmount
0,O001,C001,"Ksh. 4,500"
1,O004,C003,"Ksh. 6,800"
2,O006,C005,"Ksh. 15,000"
3,O010,C008,"Ksh. 11,000"
4,O013,C006,"Ksh. 4,300"
5,O015,C999,"Ksh. 9,900"


Electronics orders above Ksh. 8,000


,OrderID,CustomerID,OrderAmount
0,O006,C005,"Ksh. 15,000"
1,O010,C008,"Ksh. 11,000"
2,O015,C999,"Ksh. 9,900"


## Part F: `GROUP BY` and Aggregate Functions

**Q7.**

1. Electronics generated the highest category sales, `Ksh. 51,500` from 6 orders. Its average order value was `Ksh. 8,583.3333`, above the average order value for Home and close to Office.
2. Home generated the lowest sales, `Ksh. 19,500`, and the lowest average order value, `Ksh. 4,875.0000`. Office produced `Ksh. 42,500` from 5 orders and an average order value of `Ksh. 8,500.0000`, close to Electronics despite having one fewer order.

In [6]:
category_query = """
SELECT ProductCategory AS "Product Category",
       COUNT(*) AS "Number of Orders",
       SUM(OrderAmount) AS total_sales,
       AVG(OrderAmount) AS average_order_value
FROM orders
GROUP BY ProductCategory
ORDER BY CASE ProductCategory
           WHEN 'Electronics' THEN 1
           WHEN 'Home' THEN 2
           WHEN 'Office' THEN 3
           ELSE 4
         END;
"""
category_kpis_raw = pd.read_sql_query(category_query, conn)
category_kpis = category_kpis_raw.rename(columns={
    "total_sales": "Total Sales",
    "average_order_value": "Average Order Value"
}).copy()
category_kpis["Total Sales"] = category_kpis["Total Sales"].map(lambda x: f"Ksh. {x:,.0f}")
category_kpis["Average Order Value"] = category_kpis["Average Order Value"].map(lambda x: f"Ksh. {x:,.4f}")
display(category_kpis)

,Product Category,Number of Orders,Total Sales,Average Order Value
0,Electronics,6,"Ksh. 51,500","Ksh. 8,583.3333"
1,Home,4,"Ksh. 19,500","Ksh. 4,875.0000"
2,Office,5,"Ksh. 42,500","Ksh. 8,500.0000"


**Q7. Observations.**

1. Electronics led category revenue at `Ksh. 51,500` from 6 orders; its mean order value was `Ksh. 8,583.3333`. It generated about 45.4% of total recorded order value (`Ksh. 113,500`).
2. Home had the lowest revenue (`Ksh. 19,500`) and average order value (`Ksh. 4,875.0000`). Office recorded `Ksh. 42,500` across 5 orders, with an average (`Ksh. 8,500.0000`) close to Electronics despite one fewer order. These are revenue and order value comparisons only; margin and product cost are unavailable.

## Part G: Join Customers and Orders

**Q8.** No. The source has 15 order rows, but the original left join returns 16 rows. The additional row is caused by `C004`: its single order, `O005`, matches both `C004` records in the customer table. The unmatched order for `C999` is still retained because orders are on the left side of the join.

**Q9.** For `O015` and `C999`, `CustomerName` and `Segment` are NULL. In the pandas output they appear as `NaN`.

In [7]:
join_query = """
SELECT o.OrderID,
       o.CustomerID,
       c.CustomerName,
       c.Segment,
       o.OrderAmount
FROM orders AS o
LEFT JOIN customers AS c
       ON o.CustomerID = c.CustomerID
ORDER BY o.OrderID, c.CustomerName;
"""
joined = pd.read_sql_query(join_query, conn)
join_count_check = pd.DataFrame([
    {"Measure": "Order rows before join", "Value": len(orders)},
    {"Measure": "Rows after original LEFT JOIN", "Value": len(joined)},
    {"Measure": "Additional rows from join", "Value": len(joined) - len(orders)}
])
display(join_count_check)
joined_display = joined.copy()
joined_display["OrderAmount"] = joined_display["OrderAmount"].map(lambda x: f"Ksh. {x:,.0f}")
display(joined_display)

,Measure,Value
0,Order rows before join,15
1,Rows after original LEFT JOIN,16
2,Additional rows from join,1


,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,"Ksh. 4,500"
1,O002,C002,Brian,Corporate,"Ksh. 12,500"
2,O003,C001,Amina,Retail,"Ksh. 3,200"
3,O004,C003,Carol,Retail,"Ksh. 6,800"
4,O005,C004,David,SME,"Ksh. 9,100"
5,O005,C004,David,SME,"Ksh. 9,100"
6,O006,C005,Esther,Corporate,"Ksh. 15,000"
7,O007,C006,Faith,SME,"Ksh. 7,600"
8,O008,C002,Brian,Corporate,"Ksh. 8,300"
9,O009,C007,George,Retail,"Ksh. 2,500"


**Q8.** No. There are 15 orders before joining and 16 output rows afterwards. The excess row is caused by `C004`: the order table has one O005 transaction for this ID, but the customer table has two matching rows, so O005 is returned twice. The left join retains C999 as an unmatched row.

**Q9.** For O015 / C999, `CustomerName` and `Segment` are NULL (displayed as `NaN` in the pandas result).

## Part H: Reconcile the Totals

**Q10.** No. Sales total `Ksh. 113,500` before the join and `Ksh. 122,600` after the original join.

**Q11.** `C004` caused the increase. Its `O005` order, worth `Ksh. 9,100`, was counted twice.

**Q12.** Comparing row counts and sales totals checks whether the join preserved the order records and their values. Without this check, a query can run successfully while duplicated or missing records produce incorrect business results.

In [8]:
source_sales = int(orders["OrderAmount"].sum())
joined_sales = int(joined["OrderAmount"].sum())
reconciliation = pd.DataFrame([
    {"Stage": "Orders before join", "Rows": len(orders), "Sales": source_sales},
    {"Stage": "After original LEFT JOIN", "Rows": len(joined), "Sales": joined_sales},
    {"Stage": "Change", "Rows": len(joined) - len(orders), "Sales": joined_sales - source_sales}
])
reconciliation["Sales"] = reconciliation["Sales"].map(lambda x: f"Ksh. {x:,.0f}")
display(reconciliation)

,Stage,Rows,Sales
0,Orders before join,15,"Ksh. 113,500"
1,After original LEFT JOIN,16,"Ksh. 122,600"
2,Change,1,"Ksh. 9,100"


**Q10.** No. Sales increase from `Ksh. 113,500` before the join to `Ksh. 122,600` after it.

**Q11.** `C004` causes the increase: O005 (`Ksh. 9,100`) appears twice because the customer key occurs twice.

**Q12.** Row counts and control totals test whether integration has preserved the transaction population and its value. Without reconciliation, a syntactically valid join can silently duplicate or omit records and produce materially incorrect management KPIs.

## Part I: Correct the Duplicate Key

**Q13.** Yes. The cleaned join returns 15 rows, matching the number of source orders, and preserves the source total of `Ksh. 113,500`. Making `CustomerID` unique prevents `O005` from being repeated. The left join still retains `C999`, so the correction does not discard the unmatched order.

In [9]:
customers_clean = customers.drop_duplicates(subset="CustomerID", keep="first").copy()
assert customers_clean["CustomerID"].is_unique
customers_clean.to_sql("customers_clean", conn, if_exists="replace", index=False)

clean_join_query = """
SELECT o.OrderID,
       o.CustomerID,
       c.CustomerName,
       c.Segment,
       o.OrderAmount
FROM orders AS o
LEFT JOIN customers_clean AS c
       ON o.CustomerID = c.CustomerID
ORDER BY o.OrderID;
"""
clean_joined = pd.read_sql_query(clean_join_query, conn)
clean_rows = len(clean_joined)
clean_sales = int(clean_joined["OrderAmount"].sum())
clean_reconciliation = pd.DataFrame([
    {"Validation": "Order row count", "Source orders": len(orders), "Clean-join result": clean_rows, "Pass": clean_rows == len(orders)},
    {"Validation": "Sales total", "Source orders": f"Ksh. {source_sales:,.0f}", "Clean-join result": f"Ksh. {clean_sales:,.0f}", "Pass": clean_sales == source_sales}
])
display(clean_reconciliation)
assert clean_rows == len(orders)
assert clean_sales == source_sales

,Validation,Source orders,Clean-join result,Pass
0,Order row count,15,15,True
1,Sales total,"Ksh. 113,500","Ksh. 113,500",True


**Q13.** Yes. The cleaned left join returns 15 rows and `Ksh. 113,500`, equal to the source row count and sales total. The corrected key restores one customer match per ID; C999 remains unmatched but its order and value remain in the result.

## Part J: Customer Segment KPIs

**Q14.** Corporate generated the highest segment sales at `Ksh. 46,800` from 4 orders, and its average order value was `Ksh. 11,700.0000`. Retail had the largest known segment order count, 6, and sales of `Ksh. 29,600`, while its average order value was `Ksh. 4,933.3333`. SME generated `Ksh. 27,200` from 4 orders, and the separate unmatched row shows `Ksh. 9,900` that cannot yet be attributed to a segment.

In [10]:
segment_query = """
SELECT COALESCE(c.Segment, 'Unmatched / NULL') AS Segment,
       COUNT(*) AS Orders,
       SUM(o.OrderAmount) AS total_sales,
       AVG(o.OrderAmount) AS average_order_value
FROM orders AS o
LEFT JOIN customers_clean AS c
       ON o.CustomerID = c.CustomerID
GROUP BY c.Segment
ORDER BY CASE WHEN c.Segment IS NULL THEN 4
              WHEN c.Segment = 'Corporate' THEN 1
              WHEN c.Segment = 'Retail' THEN 2
              WHEN c.Segment = 'SME' THEN 3
              ELSE 5 END;
"""
segment_kpis_raw = pd.read_sql_query(segment_query, conn)
segment_kpis = segment_kpis_raw.rename(columns={
    "total_sales": "Total Sales",
    "average_order_value": "Average Order Value"
}).copy()
segment_kpis["Total Sales"] = segment_kpis["Total Sales"].map(lambda x: f"Ksh. {x:,.0f}")
segment_kpis["Average Order Value"] = segment_kpis["Average Order Value"].map(lambda x: f"Ksh. {x:,.4f}")
display(segment_kpis)

,Segment,Orders,Total Sales,Average Order Value
0,Corporate,4,"Ksh. 46,800","Ksh. 11,700.0000"
1,Retail,6,"Ksh. 29,600","Ksh. 4,933.3333"
2,SME,4,"Ksh. 27,200","Ksh. 6,800.0000"
3,Unmatched / NULL,1,"Ksh. 9,900","Ksh. 9,900.0000"


**Q14.** Corporate accounts for the highest attributed segment sales (`Ksh. 46,800` across 4 orders) and the highest average order value (`Ksh. 11,700.0000`). Retail has the largest known segment order count (6) and `Ksh. 29,600` in sales, but the lowest known segment average order value (`Ksh. 4,933.3333`). SME contributes `Ksh. 27,200` from 4 orders, while C999 remains unclassified (`Ksh. 9,900`); management should check the C999 customer reference before using the segment results.

## Part K: Merge the JSON Region Lookup

**Q15.** Yes. The merge adds a readable region name for each of the 10 distinct customers: Nairobi, Mombasa, Kisumu, or Nakuru.

**Q16.** A left merge keeps the customer record even when its `RegionCode` is absent from the JSON lookup. `RegionName` would be NULL or `NaN`, which should be investigated.

In [11]:
regions_df = pd.read_json("regions.json")
customers_regions = customers_clean.merge(
    regions_df,
    on="RegionCode",
    how="left",
    validate="many_to_one"
)
display(customers_regions[["CustomerID", "CustomerName", "Segment", "RegionCode", "RegionName"]])
assert customers_regions["RegionName"].notna().all()

,CustomerID,CustomerName,Segment,RegionCode,RegionName
0,C001,Amina,Retail,NRB,Nairobi
1,C002,Brian,Corporate,MSA,Mombasa
2,C003,Carol,Retail,NRB,Nairobi
3,C004,David,SME,KSM,Kisumu
4,C005,Esther,Corporate,NKR,Nakuru
5,C006,Faith,SME,NRB,Nairobi
6,C007,George,Retail,MSA,Mombasa
7,C008,Halima,Corporate,KSM,Kisumu
8,C009,Ian,Retail,NKR,Nakuru
9,C010,Jane,SME,NRB,Nairobi


**Q15.** Confirmed. Each of the 10 distinct customer IDs in `customers_clean` has a readable `RegionName`: Nairobi, Mombasa, Kisumu, or Nakuru.

**Q16.** With a left merge, an unrecognised RegionCode would not remove the customer row; its `RegionName` would be missing (`NaN`/NULL). That exception should be investigated.

## Final Challenge: Regional Sales Analysis

**Q17.** Nairobi generated the highest identified regional sales, `Ksh. 39,800` from 7 orders. The unmatched `C999` order is shown separately and is not included in the sales for any named region.

**Q18.** Management should not call Nairobi the best performing region on sales alone. The data contains no cost or profit figures, so regional profitability cannot be compared. It also covers only 15 orders and gives no market size or time period for assessing relative performance or trend. In addition, the `Ksh. 9,900` order for `C999` has no customer match and cannot be assigned to a region.

In [12]:
order_region = orders.merge(
    customers_clean[["CustomerID", "RegionCode"]],
    on="CustomerID",
    how="left",
    validate="many_to_one"
).merge(
    regions_df,
    on="RegionCode",
    how="left",
    validate="many_to_one"
)
order_region["Region"] = order_region["RegionName"].fillna("Unmatched / NULL")
regional_order = ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Unmatched / NULL"]
regional_kpis_raw = (order_region
    .groupby("Region", as_index=False, dropna=False)
    .agg(NumberOfOrders=("OrderID", "count"), TotalSales=("OrderAmount", "sum")))
regional_kpis_raw["Region"] = pd.Categorical(regional_kpis_raw["Region"], categories=regional_order, ordered=True)
regional_kpis_raw = regional_kpis_raw.sort_values("Region")
regional_kpis = regional_kpis_raw.copy()
regional_kpis["Total Sales"] = regional_kpis["TotalSales"].map(lambda x: f"Ksh. {x:,.0f}")
regional_kpis = regional_kpis.rename(columns={"NumberOfOrders": "Number of Orders"})[["Region", "Number of Orders", "Total Sales"]]
display(regional_kpis)
assert int(regional_kpis_raw["TotalSales"].sum()) == source_sales

,Region,Number of Orders,Total Sales
2,Nairobi,7,"Ksh. 39,800"
1,Mombasa,3,"Ksh. 23,300"
0,Kisumu,2,"Ksh. 20,100"
3,Nakuru,2,"Ksh. 20,400"
4,Unmatched / NULL,1,"Ksh. 9,900"


**Q17.** Nairobi has the highest sales among the four identified regions: 7 orders totalling `Ksh. 39,800`.

**Q18.** No. The data has no costs, so profit cannot be compared. It also covers only 15 orders and has no market size or time period for comparison. One order worth `Ksh. 9,900` cannot be assigned to a region.

## Short Conclusion

The duplicate `C004` record was the main data quality issue and caused the original join to count `O005` twice, overstating sales by `Ksh. 9,100`. After retaining one `C004` record, the join returned all 15 orders and reconciled to `Ksh. 113,500`, while the unmatched `C999` order remained visible. Sales totals must be checked after a join because duplicate keys can inflate reported results even when the SQL query runs correctly. Corporate generated the highest segment sales at `Ksh. 46,800`. The analysis does not include costs or market size, and one order cannot be assigned to a known region.

## Submission Check
Restart the kernel and run every cell from top to bottom before submitting. The notebook creates the source files and SQLite database in its working directory.